<a href="https://github.com/N3iKos/SMFactory">
  <img alt="GitHub repo" src="https://img.shields.io/badge/GitHub-6e5494?style=for-the-badge&logo=github&logoColor=white"/>
</a><br>

*   get your civitai api key from [here](https://civitai.com/user/account)

In [ ]:
# @title <b><font color='orange'>WebUI Installer</font></b> {"display-mode":"form"}

Webui = 'Forge-Neo' # @param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]
Civitai___Key = '' # @param {type:"string", placeholder:"Get Civitai key: https://civitai.com/user/account"}
HF_Read_Token = '' # @param {type:"string", placeholder:"Get HF token: https://huggingface.co/settings/tokens"}
Mount__GDrive = 'No' # @param ["No", "Yes"]

mount = Mount__GDrive

if mount == 'Yes':
    from google.colab import drive
    drive.mount('/content/drive')

!curl -sLo /content/setup.py https://github.com/N3iKos/SMFactory/raw/main/script/KC/setup.py
%run /content/setup.py --webui="$Webui" --civitai_key="$Civitai___Key" --hf_read_token="$HF_Read_Token"

from pathlib import Path
import os, shutil

SMFACTORY_DRIVE_ROOT = Path('/content/drive/MyDrive/SMFactory')

def smfactory_link_or_prepare(local_dir, drive_subdir, enabled=False):
    local_dir = Path(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)
    if not enabled:
        return local_dir
    drive_dir = SMFACTORY_DRIVE_ROOT / drive_subdir
    drive_dir.mkdir(parents=True, exist_ok=True)
    link = local_dir / f'drive-{drive_subdir}'
    if link.exists() or link.is_symlink():
        return drive_dir
    link.symlink_to(drive_dir, target_is_directory=True)
    return drive_dir

def smfactory_download(url, target_dir, filename='', parallel=False, workers=3):
    if not url:
        return
    cmd = f"%download {url}"
    if filename:
        cmd += f" {filename}"
    cmd += f" {target_dir}"
    if parallel:
        cmd += f" --parallel --workers={workers}"
    get_ipython().run_line_magic('download', cmd[len('%download '):])


In [ ]:
# @title <b><font color='orange'>Model Downloader</font></b> {"display-mode":"form"}

Checkpoint_1 = 'https://huggingface.co/pantat88/back_up/resolve/main/bigblu25dmix25DStyle_v10.safetensors' # @param {type:"string"}
Checkpoint_2 = '' # @param {type:"string"}
Checkpoint_3 = '' # @param {type:"string"}
Checkpoint_4 = '' # @param {type:"string"}
Checkpoint_5 = '' # @param {type:"string"}
Lora_1 = 'https://civitai.com/models/122359/detail-tweaker-xl' # @param {type:"string"}
Lora_2 = 'https://civitai.com/models/669571/pony-add-more-details' # @param {type:"string"}
Lora_3 = '' # @param {type:"string"}
Lora_4 = '' # @param {type:"string"}
Lora_5 = '' # @param {type:"string"}
VAE_URL = '' # @param {type:"string", placeholder:"URL or leave empty"}
Load_from_Drive = False # @param {type:"boolean"}
Parallel_Download = True # @param {type:"boolean"}
Max_Workers = 3 # @param {type:"slider", min:1, max:5, step:1}

ckpt_dir = smfactory_link_or_prepare(CKPT, 'checkpoint', Load_from_Drive)
lora_dir = smfactory_link_or_prepare(LORA, 'lora', Load_from_Drive)
vae_dir = smfactory_link_or_prepare(VAE, 'vae', Load_from_Drive)

for url in [Checkpoint_1, Checkpoint_2, Checkpoint_3, Checkpoint_4, Checkpoint_5]:
    smfactory_download(url, ckpt_dir, parallel=Parallel_Download, workers=Max_Workers)
for url in [Lora_1, Lora_2, Lora_3, Lora_4, Lora_5]:
    smfactory_download(url, lora_dir, parallel=Parallel_Download, workers=Max_Workers)
smfactory_download(VAE_URL, vae_dir, parallel=Parallel_Download, workers=Max_Workers)


In [ ]:
# @title <b><font color='orange'>Extra Assets</font></b> {"display-mode":"form"}

Extension_1 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_2 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_3 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_4 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_5 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Embedding_1 = '' # @param {type:"string"}
Embedding_2 = '' # @param {type:"string"}
Embedding_3 = '' # @param {type:"string"}
Upscaler_1 = '' # @param {type:"string"}
Upscaler_2 = '' # @param {type:"string"}
Upscaler_3 = '' # @param {type:"string"}
Load_from_Drive = False # @param {type:"boolean"}
Assets_Parallel_Download = True # @param {type:"boolean"}
Assets_Max_Workers = 3 # @param {type:"slider", min:1, max:5, step:1}

emb_dir = smfactory_link_or_prepare(Embeddings, 'embeddings', Load_from_Drive)
up_dir = smfactory_link_or_prepare(Upscalers, 'upscalers', Load_from_Drive)

extensions = [u for u in [Extension_1, Extension_2, Extension_3, Extension_4, Extension_5] if u.strip()]
if extensions:
    Path('/tmp/smfactory_extensions.txt').write_text('\n'.join(extensions), encoding='utf-8')
    %cd -q $Extensions
    %clone /tmp/smfactory_extensions.txt

for url in [Embedding_1, Embedding_2, Embedding_3]:
    smfactory_download(url, emb_dir, parallel=Assets_Parallel_Download, workers=Assets_Max_Workers)
for url in [Upscaler_1, Upscaler_2, Upscaler_3]:
    smfactory_download(url, up_dir, parallel=Assets_Parallel_Download, workers=Assets_Max_Workers)


In [ ]:
# @title <b><font color='orange'>FLUX Model Downloader</font></b> {"display-mode":"form"}

FLUX_Variant = 'FLUX.1-schnell' # @param ["FLUX.1-schnell", "FLUX.1-dev", "Custom URLs"]
FLUX_Unet = '' # @param {type:"string", placeholder:"Custom Unet URL"}
FLUX_Clip_L = '' # @param {type:"string", placeholder:"Custom Clip L URL"}
FLUX_T5XXL = '' # @param {type:"string", placeholder:"Custom T5XXL URL"}
FLUX_VAE = '' # @param {type:"string", placeholder:"Custom VAE URL"}
Load_from_Drive = False # @param {type:"boolean"}
Parallel_FLUX_Download = True # @param {type:"boolean"}
FLUX_Max_Workers = 2 # @param {type:"slider", min:1, max:3, step:1}

FLUX_DEFAULTS = {
    'FLUX.1-schnell': {
        'unet': 'https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/flux1-schnell.safetensors',
        'clip_l': '',
        't5xxl': '',
        'vae': ''
    },
    'FLUX.1-dev': {
        'unet': 'https://huggingface.co/black-forest-labs/FLUX.1-dev/resolve/main/flux1-dev.safetensors',
        'clip_l': '',
        't5xxl': '',
        'vae': ''
    }
}

flux_dirs = {
    'unet': smfactory_link_or_prepare(UNET, 'flux-unet', Load_from_Drive),
    'clip_l': smfactory_link_or_prepare(CLIP, 'flux-clip', Load_from_Drive),
    't5xxl': smfactory_link_or_prepare(CLIP, 'flux-t5xxl', Load_from_Drive),
    'vae': smfactory_link_or_prepare(VAE, 'flux-vae', Load_from_Drive),
}
urls = FLUX_DEFAULTS.get(FLUX_Variant, {}).copy()
custom = {'unet': FLUX_Unet, 'clip_l': FLUX_Clip_L, 't5xxl': FLUX_T5XXL, 'vae': FLUX_VAE}
for key, value in custom.items():
    if value:
        urls[key] = value
for key, url in urls.items():
    smfactory_download(url, flux_dirs[key], parallel=Parallel_FLUX_Download, workers=FLUX_Max_Workers)


In [ ]:
''' Controlnet '''
%run $Controlnet_Widget

## Launcher
args list :
-  **A1111** = `--xformers`
- **Forge** = `--disable-xformers --opt-sdp-attention --cuda-stream`
- **ReForge** = `--xformers --cuda-stream`
- **Forge-Classic** = `--xformers --cuda-stream --persistent-patches`
- **Forge-Neo** = `--xformers --cuda-malloc --cuda-stream`
- **ComfyUI** = `--dont-print-server --use-pytorch-cross-attention`
- **SwarmUI** = `--launch_mode none`
<br><br>

For ComfyUI, add `--skip-comfyui-check` to skip checking the main requirements and custom node dependencies

Add **--N=ngrok_token** to start NGROK tunnel<br>
Add **--Z=zrok_token** to start ZROK tunnel

In [ ]:
# @title <b><font color='orange'>Launcher WebUI</font></b> {"display-mode":"form"}

Software = 'Forge-Neo' # @param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]
Ngrok_Token = '' # @param {type:"string"}
Zrok_Token = '' # @param {type:"string"}
Extra_Args = '--xformers' # @param {type:"string"}
Skip_ComfyUI_Check = False # @param {type:"boolean"}
Skip_Widget = False # @param {type:"boolean"}

args = Extra_Args
if Ngrok_Token:
    args += f' --N={Ngrok_Token}'
if Zrok_Token:
    args += f' --Z={Zrok_Token}'
if Skip_ComfyUI_Check:
    args += ' --skip-comfyui-check'
if Skip_Widget:
    args += ' --skip-widget'

print('Select the same WebUI that you installed in the first cell.')
%cd -q $WebUI
%run segsmaker.py $args
